# 🚂 RailSafe — Réentraînement avec météo

**Objectif** : Intégrer les données météo Open-Meteo et voir si le ROC-AUC s'améliore.

**Nouvelles features météo :**
- `temp_mean_mois` — température moyenne mensuelle
- `precip_sum_mois` — précipitations totales
- `wind_max_mois` — vent maximum
- `snow_sum_mois` — neige totale
- `jours_pluie` — nombre de jours de pluie
- `jours_neige` — nombre de jours de neige

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, roc_auc_score,
    roc_curve, f1_score, confusion_matrix
)
from xgboost import XGBClassifier

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

RAW_DIR    = Path('../data/raw')
PROC_DIR   = Path('../data/processed')
MODELS_DIR = Path('../models')
PROC_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print('✅ Imports OK')

## 2. Chargement & Feature Engineering TGV (focus TGV pour meilleurs résultats)

In [ ]:
tgv = pd.read_csv(RAW_DIR / 'regularite_tgv.csv', sep=';', encoding='utf-8')

# Parsing date
tgv['date']      = pd.to_datetime(tgv['Date'], format='%Y-%m')
tgv['annee']     = tgv['date'].dt.year
tgv['mois']      = tgv['date'].dt.month
tgv['trimestre'] = tgv['date'].dt.quarter

# Features métier
tgv['taux_retard']     = tgv["Nombre de trains en retard à l'arrivée"] / tgv['Nombre de circulations prévues']
tgv['taux_annulation'] = tgv['Nombre de trains annulés'] / tgv['Nombre de circulations prévues']
tgv['taux_retard_30m'] = tgv['Nombre trains en retard > 30min'] / tgv['Nombre de circulations prévues']
tgv['taux_retard_60m'] = tgv['Nombre trains en retard > 60min'] / tgv['Nombre de circulations prévues']

# Liaison propre (sans anomalies)
tgv = tgv[tgv["Gare d'arrivée"].notna()]
tgv = tgv[tgv["Gare d'arrivée"] != '0']
tgv = tgv[tgv["Gare de départ"] != '0']
tgv = tgv[tgv['Nombre de circulations prévues'] > 0]
tgv['liaison'] = tgv['Gare de départ'].str.strip() + ' -> ' + tgv["Gare d'arrivée"].str.strip()

# Ville de départ pour merge météo
VILLE_MAP = {
    'PARIS'    : 'Paris',
    'LYON'     : 'Lyon',
    'MARSEILLE': 'Marseille',
    'BORDEAUX' : 'Bordeaux',
    'LILLE'    : 'Lille',
    'TOULOUSE' : 'Toulouse',
    'NANTES'   : 'Nantes',
    'RENNES'   : 'Rennes',
}

def map_ville(gare: str) -> str:
    gare = str(gare).upper()
    for key, val in VILLE_MAP.items():
        if key in gare:
            return val
    return 'Paris'  # défaut

tgv['ville_depart'] = tgv['Gare de départ'].apply(map_ville)

# Cible binaire
seuil = tgv['taux_retard'].quantile(0.75)
tgv['retard_eleve'] = (tgv['taux_retard'] > seuil).astype(int)

print(f'TGV : {tgv.shape[0]:,} lignes')
print(f'Villes départ : {tgv["ville_depart"].value_counts().to_dict()}')

## 3. Chargement & Merge météo

In [ ]:
meteo = pd.read_csv(RAW_DIR / 'meteo_villes.csv')
print(f'Météo : {meteo.shape}')
print(meteo.head())

In [ ]:
# Merge TGV + météo sur (annee, mois, ville_depart)
tgv_meteo = tgv.merge(
    meteo,
    left_on  = ['annee', 'mois', 'ville_depart'],
    right_on = ['annee', 'mois', 'ville'],
    how      = 'left'
)

print(f'Après merge : {tgv_meteo.shape}')
print(f'Lignes avec météo : {tgv_meteo["temp_mean_mois"].notna().sum():,}')
print(f'Lignes sans météo : {tgv_meteo["temp_mean_mois"].isna().sum():,}')

## 4. Corrélations météo vs retards

In [ ]:
meteo_cols = ['temp_mean_mois', 'precip_sum_mois', 'wind_max_mois',
              'snow_sum_mois', 'jours_pluie', 'jours_neige']

corrs = tgv_meteo[meteo_cols + ['taux_retard']].corr()['taux_retard'].drop('taux_retard')

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corrs.values]
corrs.plot(kind='barh', ax=ax, color=colors, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Corrélations météo vs taux de retard TGV', fontweight='bold', fontsize=13)
ax.set_xlabel('Corrélation de Pearson')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../data/viz_meteo_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nCorrélations météo :')
print(corrs.sort_values(key=abs, ascending=False))

## 5. Préparation ML avec météo

In [ ]:
# Encodage
le_liaison = LabelEncoder()
tgv_meteo['liaison_enc'] = le_liaison.fit_transform(tgv_meteo['liaison'].astype(str))

# Features SANS météo (baseline)
FEATURES_BASE = [
    'annee', 'mois', 'trimestre',
    'taux_annulation',
    'Prct retard pour causes externes',
    'Prct retard pour cause infrastructure',
    'Prct retard pour cause gestion trafic',
    'Prct retard pour cause matériel roulant',
    'liaison_enc'
]

# Features AVEC météo
FEATURES_METEO = FEATURES_BASE + [
    'temp_mean_mois', 'precip_sum_mois',
    'wind_max_mois', 'snow_sum_mois',
    'jours_pluie', 'jours_neige'
]

TARGET = 'retard_eleve'

# Dataset propre
df_clean = tgv_meteo.dropna(subset=['taux_annulation']).copy()

X_base  = df_clean[FEATURES_BASE]
X_meteo = df_clean[FEATURES_METEO]
y       = df_clean[TARGET]

# Split
X_base_tr,  X_base_te,  y_tr, y_te = train_test_split(X_base,  y, test_size=0.2, random_state=42, stratify=y)
X_meteo_tr, X_meteo_te, _,    _    = train_test_split(X_meteo, y, test_size=0.2, random_state=42, stratify=y)

print(f'Dataset TGV seul : {df_clean.shape[0]:,} lignes')
print(f'Features base  : {len(FEATURES_BASE)}')
print(f'Features météo : {len(FEATURES_METEO)}')

## 6. Comparaison modèles : Sans météo vs Avec météo

In [ ]:
PARAMS = dict(
    n_estimators          = 300,
    max_depth             = 6,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 3,
    random_state          = 42,
    eval_metric           = 'logloss',
    early_stopping_rounds = 20,
    verbosity             = 0
)

# Modèle sans météo
model_base = XGBClassifier(**PARAMS)
model_base.fit(X_base_tr, y_tr, eval_set=[(X_base_te, y_te)], verbose=False)
auc_base = roc_auc_score(y_te, model_base.predict_proba(X_base_te)[:, 1])
f1_base  = f1_score(y_te, model_base.predict(X_base_te))

# Modèle avec météo
model_meteo = XGBClassifier(**PARAMS)
model_meteo.fit(X_meteo_tr, y_tr, eval_set=[(X_meteo_te, y_te)], verbose=False)
auc_meteo = roc_auc_score(y_te, model_meteo.predict_proba(X_meteo_te)[:, 1])
f1_meteo  = f1_score(y_te, model_meteo.predict(X_meteo_te))

print('=' * 55)
print('📊 COMPARAISON — Sans météo vs Avec météo (TGV seul)')
print('=' * 55)
print(f'{'Modèle':<25} {'ROC-AUC':>10} {'F1':>10}')
print('-' * 45)
print(f'{'Sans météo':<25} {auc_base:>10.4f} {f1_base:>10.4f}')
print(f'{'Avec météo':<25} {auc_meteo:>10.4f} {f1_meteo:>10.4f}')
print(f'{'Gain AUC':<25} {auc_meteo - auc_base:>+10.4f}')
print(f'{'Gain F1':<25} {f1_meteo - f1_base:>+10.4f}')

In [ ]:
# Courbes ROC comparées
fpr_b, tpr_b, _ = roc_curve(y_te, model_base.predict_proba(X_base_te)[:, 1])
fpr_m, tpr_m, _ = roc_curve(y_te, model_meteo.predict_proba(X_meteo_te)[:, 1])

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(fpr_b, tpr_b, color='steelblue', lw=2, label=f'Sans météo — AUC={auc_base:.3f}')
ax.plot(fpr_m, tpr_m, color='coral',     lw=2, label=f'Avec météo  — AUC={auc_meteo:.3f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('Taux de faux positifs')
ax.set_ylabel('Taux de vrais positifs')
ax.set_title('ROC — Sans météo vs Avec météo (TGV)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/viz_roc_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Sauvegardé : data/viz_roc_comparison.png')

## 7. SHAP — Importance avec météo

In [ ]:
explainer   = shap.TreeExplainer(model_meteo)
shap_values = explainer.shap_values(X_meteo_te)

feature_names_fr = {
    'annee'                                    : 'Année',
    'mois'                                     : 'Mois',
    'trimestre'                                : 'Trimestre',
    'taux_annulation'                          : 'Taux annulation',
    'Prct retard pour causes externes'         : 'Cause externe (%)',
    'Prct retard pour cause infrastructure'    : 'Infrastructure (%)',
    'Prct retard pour cause gestion trafic'    : 'Gestion trafic (%)',
    'Prct retard pour cause matériel roulant'  : 'Matériel roulant (%)',
    'liaison_enc'                              : 'Liaison',
    'temp_mean_mois'                           : '🌡️ Température moy.',
    'precip_sum_mois'                          : '🌧️ Précipitations',
    'wind_max_mois'                            : '💨 Vent max',
    'snow_sum_mois'                            : '❄️ Neige',
    'jours_pluie'                              : '🌧️ Jours pluie',
    'jours_neige'                              : '❄️ Jours neige',
}
X_te_named = X_meteo_te.rename(columns=feature_names_fr)

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_te_named, plot_type='bar', show=False, color='steelblue')
plt.title('SHAP — Importance avec météo (TGV)', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../data/viz_shap_meteo.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Sauvegardé : data/viz_shap_meteo.png')

## 8. Sauvegarde du meilleur modèle

In [ ]:
# On garde le meilleur modèle
best_model    = model_meteo if auc_meteo >= auc_base else model_base
best_features = FEATURES_METEO if auc_meteo >= auc_base else FEATURES_BASE
best_name     = 'avec météo' if auc_meteo >= auc_base else 'sans météo'
best_auc      = max(auc_meteo, auc_base)
best_f1       = f1_meteo if auc_meteo >= auc_base else f1_base

model_bundle = {
    'model'        : best_model,
    'feature_cols' : best_features,
    'le_liaison'   : le_liaison,
    'le_type'      : LabelEncoder().fit(['TGV']),
    'le_region'    : LabelEncoder().fit(['National']),
    'seuil_tgv'    : seuil,
    'seuil_ter'    : 0.102,
    'seuil_ic'     : 0.183,
    'with_meteo'   : auc_meteo >= auc_base,
    'metrics'      : {
        'roc_auc'      : round(best_auc, 4),
        'f1'           : round(best_f1, 4),
        'auc_base'     : round(auc_base, 4),
        'auc_meteo'    : round(auc_meteo, 4),
        'gain_meteo'   : round(auc_meteo - auc_base, 4),
    }
}

joblib.dump(model_bundle, MODELS_DIR / 'railsafe_model.joblib')
print(f'✅ Meilleur modèle sauvegardé : {best_name}')

print(f"""
{'='*55}
📊 RÉSUMÉ FINAL
{'='*55}
Meilleur modèle : XGBoost {best_name}
ROC-AUC         : {best_auc:.4f}
F1-Score        : {best_f1:.4f}
Gain météo AUC  : {auc_meteo - auc_base:+.4f}
Features        : {len(best_features)}
""")